In [1]:
from pyspark.sql import SparkSession, functions as F, Window

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("main") \
    .config("spark.ui.enabled", "true") \
    .config("spark.ui.port", "4040") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("fs.s3a.endpoint", "http://minio:9000") \
    .config("fs.s3a.access.key", "minioadmin") \
    .config("fs.s3a.secret.key", "minioadmin") \
    .config("fs.s3a.path.style.access", "true") \
    .config("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

In [3]:
users = spark.createDataFrame(
    [ ("u1", "Berlin"),
    ("u2", "Berlin"),
    ("u3", "Munich"),
    ("u4", "Hamburg"), ],
    ["user_id", "city"] 
)
orders = spark.createDataFrame(
    [ ("o1", "u1", "p1", 2, 10.0),
    ("o2", "u1", "p2", 1, 30.0),
    ("o3", "u2", "p1", 1, 10.0),
    ("o4", "u2", "p3", 5, 7.0),
    ("o5", "u3", "p2", 3, 30.0),
    ("o6", "u3", "p3", 1, 7.0),
    ("o7", "u4", "p1", 10, 10.0), ],
    ["order_id", "user_id", "product_id", "qty", "price"] 
)
products = spark.createDataFrame(
    [ ("p1", "Ring VOLA"),
    ("p2", "Ring POROG"),
    ("p3", "Ring TISHINA"), ],
    ["product_id", "product_name"] 
)

In [4]:
win = Window.partitionBy("city", "product_id", "product_name")
top_2_win = Window.partitionBy("city").orderBy("revenue_sum")

In [5]:
mart_city_top_products = users \
    .join(orders, on="user_id", how="inner") \
    .join(products, on="product_id", how="inner") \
    .withColumn("revenue", F.col("qty") * F.col("price")) \
    .withColumn("orders_cnt", F.count("*").over(win)) \
    .withColumn("qty_sum", F.sum("qty").over(win)) \
    .withColumn("revenue_sum", F.sum("revenue").over(win)) \
    .withColumn("top_revenuesum_by_city", F.dense_rank().over(top_2_win)) \
    .filter(F.col("top_revenuesum_by_city") == 2)

In [7]:
mart_city_top_products.write.mode("overwrite").parquet("s3a://spark-data/mart_city_top_products/")

26/02/26 18:12:29 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [12]:
mart_city_top_products.show(5)

+----------+-------+------+--------+---+-----+------------+-------+----------+-------+-----------+----------------------+
|product_id|user_id|  city|order_id|qty|price|product_name|revenue|orders_cnt|qty_sum|revenue_sum|top_revenuesum_by_city|
+----------+-------+------+--------+---+-----+------------+-------+----------+-------+-----------+----------------------+
|        p3|     u2|Berlin|      o4|  5|  7.0|Ring TISHINA|   35.0|         1|      5|       35.0|                     2|
|        p2|     u3|Munich|      o5|  3| 30.0|  Ring POROG|   90.0|         1|      3|       90.0|                     2|
+----------+-------+------+--------+---+-----+------------+-------+----------+-------+-----------+----------------------+



In [8]:
df = spark.read.parquet("s3a://spark-data/mart_city_top_products/")

In [9]:
df.show(10)

+----------+-------+------+--------+---+-----+------------+-------+----------+-------+-----------+----------------------+
|product_id|user_id|  city|order_id|qty|price|product_name|revenue|orders_cnt|qty_sum|revenue_sum|top_revenuesum_by_city|
+----------+-------+------+--------+---+-----+------------+-------+----------+-------+-----------+----------------------+
|        p3|     u2|Berlin|      o4|  5|  7.0|Ring TISHINA|   35.0|         1|      5|       35.0|                     2|
|        p2|     u3|Munich|      o5|  3| 30.0|  Ring POROG|   90.0|         1|      3|       90.0|                     2|
+----------+-------+------+--------+---+-----+------------+-------+----------+-------+-----------+----------------------+

